# BTL02 — Đánh giá Baseline CNN

Phần của Xuân Anh: metric, confusion matrix, ảnh đúng/sai và filter conv3.
Phần A phân tích bằng chứng đã lưu của Nhi, không cần checkpoint. Phần B đánh giá lại khi có checkpoint và ảnh; không train lại.

Notebook baseline chọn checkpoint trên cùng tập test sau mỗi epoch, nên các chỉ số đó không phải test độc lập.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "evaluate.py").is_file():
    raise RuntimeError("Mo notebook tu thu muc HocSau-python hoac notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from src.evaluate import evaluate
print("Project:", ROOT)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


## Phần A Phân tích kết quả đã lưu

49 số đếm được chép từ ảnh confusion matrix trong repo, kiểm tra bằng SHA-256 và đối chiếu chỉ số làm tròn trong notebook gốc. Đây không phải lần inference mới và không khôi phục được danh tính từng ảnh.

In [ ]:
from src.analyze_saved_results import analyze

saved_metrics = analyze()
print("Accuracy:", "{:.2%}".format(saved_metrics["accuracy"]))
print("Macro-F1:", "{:.4f}".format(saved_metrics["macro_f1"]))
print("Correct / total:", saved_metrics["correct"], "/", saved_metrics["samples"])


In [ ]:
from IPython.display import Image, display

for filename in ["baseline_learning_curve.png", "baseline_confusion_matrix.png"]:
    display(Image(filename=str(ROOT / "results" / "plots" / filename), width=850))


## Phần B Đánh giá từ checkpoint

Chỉ chạy tiếp khi đã nhận checkpoint và bộ ảnh. Nếu chưa có, có thể dừng ở đây và đọc báo cáo cùng bảng metric trong `results/evaluation/saved_baseline`.

## 1. Kiểm tra đầu vào

Đặt checkpoint của Nhi vào `checkpoints/baseline_best.pth`; đặt dữ liệu vào `data/raw/test/<nhãn>/*`. Có thể sửa đường dẫn bên dưới. Thứ tự nhãn được kiểm tra theo ImageFolder, không dùng thứ tự mã nhãn FER2013 gốc trong README.

In [ ]:
CHECKPOINT = ROOT / "checkpoints" / "baseline_best.pth"
DATA_DIR = ROOT / "data" / "raw" / "test"
OUTPUT_DIR = ROOT / "results" / "evaluation" / "baseline"

missing = []
if not CHECKPOINT.is_file():
    missing.append(f"Checkpoint: {CHECKPOINT}")
if not DATA_DIR.is_dir():
    missing.append(f"Du lieu: {DATA_DIR}")
if missing:
    raise FileNotFoundError("Can nhan file tu Nhi truoc khi chay:\n" + "\n".join(missing))
print("Da co checkpoint va thu muc du lieu.")


## 2. Chạy đánh giá

CPU phù hợp cho bước này. Không augmentation; chuẩn hóa ảnh giống baseline. Chỉ xuất kết quả sau khi inference thành công.

In [ ]:
metrics = evaluate(
    checkpoint=CHECKPOINT,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    device="cpu",
    batch_size=128,
    split_name="test",
    selection_used=True,
)


## 3. Xem biểu đồ và các trường hợp đúng/sai

In [ ]:
from IPython.display import display, Image, Markdown

for filename in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "true_predict_images.png",
    "wrong_predict_images.png",
    "final_conv_filters.png",
]:
    print(filename)
    display(Image(filename=str(OUTPUT_DIR / filename), width=900))
display(Markdown((OUTPUT_DIR / "evaluation_summary.md").read_text(encoding="utf-8")))


## 4. Gợi ý viết Mục 3 của báo cáo

1. Ghi rõ checkpoint, tập đánh giá, số ảnh, tiền xử lý và giới hạn về việc chọn checkpoint.
2. Trình bày Accuracy, Macro-F1, Weighted-F1 và precision/recall từng lớp từ file xuất.
3. Phân tích các cặp nhãn nhầm nhiều, đối chiếu số lượng mẫu từng lớp.
4. Xem ảnh đúng/sai trước khi đề xuất nguyên nhân; phân biệt quan sát và giả thuyết.
5. Hình filter lấy trung bình trọng số conv3 qua các kênh đầu vào; không gọi là feature map hoặc Grad-CAM.

Số liệu phải lấy từ lần chạy thật. Chi tiết: `docs/EVALUATION_GUIDE.md`.